In [1]:
library(CellChat) # 2.1.2
library(Matrix)
library(Seurat)
library(patchwork)
library(BiocParallel)
library(data.table)
library(dplyr)
library(readr)
library(ggplot2)
library(gridExtra)

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: igraph

Warning message:
“package ‘igraph’ was built under R version 4.3.3”

Attaching package: ‘igraph’


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union


Loading required package: ggplot2

Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘Seurat’ was built under R version 4.3.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.3.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.3.3”

Attaching pack

In [3]:
## Load RNA matrix and convert into cellchat objects
sample_meta_path <- "../input_metadata/sample_data.txt"
meta <- fread(sample_meta_path, sep = "\t", data.table = FALSE)

required_cols <- c("sample_id", "RNA_matrix")
missing <- setdiff(required_cols, colnames(meta))
if (length(missing) > 0) {
  stop(paste("Missing required columns:", paste(missing, collapse = ", ")))
}

seurat_list <- list()
print("start loading")
for (i in 1:nrow(meta)) {
  sample_id <- meta$sample_id[i]
  data_dir <- meta$RNA_matrix[i]

  counts <- Read10X(data.dir = data_dir)

  seu <- CreateSeuratObject(counts = counts, project = sample_id)

  seu$sample <- sample_id

  colnames(seu) <- paste0(sample_id, "_", colnames(seu))

  seurat_list[[sample_id]] <- seu
}

combined <- Reduce(function(x, y) merge(x, y), seurat_list)

combined[["percent.mt"]] <- PercentageFeatureSet(combined, pattern = "^MT-")

qc_before <- combined@meta.data %>%
  group_by(sample) %>%
  summarise(
    n_cells_before = n(),
    median_genes_before = median(nFeature_RNA),
    median_mt_before = median(percent.mt)
  )

combined <- subset(
  combined,
  subset = nFeature_RNA > 200 &
           nFeature_RNA < 5000 &
           percent.mt < 20
)

qc_after <- combined@meta.data %>%
  group_by(sample) %>%
  summarise(
    n_cells_after = n(),
    median_genes_after = median(nFeature_RNA),
    median_mt_after = median(percent.mt)
  )

qc_summary <- left_join(qc_before, qc_after, by = "sample") %>%
  mutate(
    pct_cells_retained = n_cells_after / n_cells_before * 100
  )
print("start Normalizing")
seurat_list <- lapply(seurat_list, NormalizeData)
combined <- Reduce(function(x, y) merge(x, y), seurat_list)

combined <- JoinLayers(combined)
                   
print("start GetAssayData")

data_input <- GetAssayData(combined, slot = "data")

meta_input <- combined@meta.data

saveRDS(data_input, file = "tmp_dataset/cellchat_input_matrix.rds")
saveRDS(meta_input, file = "tmp_dataset/cellchat_meta.rds")

[1] "start loading"
[1] "start Normalizing"


Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

N

[1] "start GetAssayData"


In [2]:
## load RDS
data_input <- readRDS("tmp_dataset/cellchat_input_matrix.rds")
meta_input <- readRDS("tmp_dataset/cellchat_meta.rds")
cell_meta <- read_tsv("../input_metadata//cell_meta.tsv")

cell_meta <- cell_meta %>%
  filter(cell_type != "Schwann & other") ## exclude small cell type cluster from analysis

cell_meta$cell_type_large <-cell_meta$cell_type
cell_meta <- cell_meta %>%
  mutate(
    cell_type_large = coalesce(
      if_else(!is.na(epi_cl), paste0("epi_", epi_cl), NA_character_),
      if_else(!is.na(t_cl), paste0("t_", t_cl), NA_character_),
      if_else(!is.na(b_cl), paste0("b_", b_cl), NA_character_),
      if_else(!is.na(end_cl), paste0("end_", end_cl), NA_character_),
      if_else(!is.na(smc_cl), paste0("smc_", smc_cl), NA_character_),
      if_else(!is.na(mye_cl), paste0("mye_", mye_cl), NA_character_),
      if_else(!is.na(fib_cl), paste0("fib_", fib_cl), NA_character_),
      cell_type_large
    )
  ) %>% ## filter out outlier cells in each cluster
  filter(!cell_type_large %in% c("B","Endothelial","Myeloid","Plasma","Smooth muscle","T"))

cells_use <- intersect(colnames(data_input), cell_meta$cell_id)
data_input <- data_input[, cells_use, drop = FALSE]
meta_input <- meta_input[cells_use, , drop = FALSE]
cell_meta <- cell_meta[match(cells_use, cell_meta$cell_id), ]
meta_input$cell_type_large <- cell_meta$cell_type_large
meta_input$group <- paste0("G",cell_meta$Group)

celltype_order <- names(table(meta_input$cell_type_large))
meta_input$labels <- factor(meta_input$cell_type_large ,levels = celltype_order)


Rows: 437125 Columns: 28
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (20): cell_id, sample_id, cell_type, id, epi_cluster, t_cluster, fib_clu...
dbl  (8): hypo_frac, UMAP_1, UMAP_2, TSNE_1, TSNE_2, rna_cluster, Group, mc_...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
### loop over 3 groups, create interaction network
groups <- unique(meta_input$Group)

cellchat_list <- list()
df_list <- list()

for (g in groups) {

  message("Processing Group: ", g)

  cells_use <- rownames(meta_input)[meta_input$group == g]

  meta_sub <- meta_input[cells_use, , drop = FALSE]
  data_sub <- data_input[, cells_use, drop = FALSE]
  stopifnot(identical(rownames(meta_sub), colnames(data_sub)))

  cell_counts <- table(meta_sub$labels)
  valid_labels <- names(cell_counts[cell_counts >= 10])

  meta_sub <- meta_sub[meta_sub$labels %in% valid_labels, ]
  data_sub <- data_sub[, rownames(meta_sub)]

  if (length(unique(meta_sub$labels)) < 2) {
    message("Skipping Group ", g, " (not enough cell types)")
    next
  }

  ordered_idx <- order(meta_sub$labels)
  meta_sub <- meta_sub[ordered_idx, ]
  data_sub <- data_sub[, ordered_idx]

  cellchat <- createCellChat(
    object = data_sub,
    meta = meta_sub,
    group.by = "labels"
  )

  cellchat@DB <- CellChatDB.human

  cellchat <- subsetData(cellchat)

  cellchat <- identifyOverExpressedGenes(cellchat, do.fast = FALSE)
  cellchat <- identifyOverExpressedInteractions(cellchat)

  cellchat <- smoothData(cellchat, adj = PPI.human)

  cellchat <- computeCommunProb(
    cellchat,
    type = "triMean",
    raw.use = FALSE,
    nboot = 20
  )

  cellchat <- filterCommunication(cellchat, min.cells = 10)

  cellchat <- computeCommunProbPathway(cellchat)

  cellchat <- netAnalysis_computeCentrality(cellchat, slot.name = "netP")

  cellchat <- aggregateNet(cellchat)

  df.net <- subsetCommunication(cellchat)

  cellchat_list[[g]] <- cellchat
  df_list[[g]] <- df.net

  saveRDS(cellchat, paste0("tmp_dataset/RDS/cellchat_large_", g, ".rds"))
  saveRDS(df.net, paste0("tmp_dataset/RDS/df.net_large_", g, ".rds"))
  write.csv(df.net, paste0("tmp_dataset/RDS/df.net_large_", g, ".csv"), row.names = FALSE)
}

In [ ]:
cellchat_list<-list()
for(condition in c("G1","G2")) ## change to G1, G3 or G2, G3 for pairwise comparison
{
    cellchat_list[[condition]] <- readRDS(paste0("RDS/cellchat_large_",condition,".rds"))
}

cellchat_merged <- mergeCellChat(
  object.list = cellchat_list,  
  add.names = names(cellchat_list),  
  merge.data = TRUE,  
  cell.prefix = FALSE  
)
all_pathways <- unique(unlist(lapply(cellchat_list, function(x) x@netP$pathways)))
cellchat_merged@netP$pathways <- all_pathways

cellchat_merged@meta$labels <- factor(cellchat_merged@meta$labels, levels = 
   c("epi_LE","epi_BE","epi_C",
     "end_vEC","end_angioEC","end_aEC","end_EndMT",
     "fib_uCAF","fib_iCAF","fib_POSTN_p_myCAF","fib_POSTN_m_myCAF",
     "smc_cSMC","smc_Pericyte","smc_sSMC",
     "t_Naive","t_CD8_T","t_TRM","t_T_cluster_4","t_Treg","t_NKT","t_dgT","t_exhausted",
     "mye_TAM","mye_Monocyte","mye_DC",
     "b_Naive","b_Activated","b_Memory","b_Plasma",
    "Mast")
) ## reorder labels for displaying

gg1 <- compareInteractions(cellchat_merged, show.legend = F, 
                           group = c(1:2))
gg2 <- compareInteractions(cellchat_merged, show.legend = F, 
                           group = c(1:2),measure = "weight")
p_ptm_number_strength<-gg1 + gg2
#p_ptm_number_strength
ggsave("figures/cellchat/G1G2_number_strength.pdf",
       plot = p_ptm_number_strength,
       width = 8,          
       height = 4.5,       
       dpi = 300)

netVisual_diffInteraction(cellchat_merged, weight.scale = T)

netVisual_diffInteraction(cellchat_merged, 
                          weight.scale = T, 
                        measure = "weight")
gg1 <- netVisual_heatmap(cellchat_merged)
gg2 <- netVisual_heatmap(cellchat_merged, measure = "weight")
pdf("figures/cellchat/G1G2_RNR_heatmap_1.pdf", width = 8, height = 4.5)
gg1
dev.off()
pdf("figures/cellchat/G1G2_RNR_heatmap_2.pdf", width = 8, height = 4.5)
gg2
dev.off()
weight.max <- getMaxWeight(cellchat_list, attribute = c("idents","count"))
par(mfrow = c(1,2), xpd=TRUE)
for (i in 1:length(cellchat_list)) {
  netVisual_circle(cellchat_list[[i]]@net$count, weight.scale = T, 
                   label.edge= F, edge.weight.max = weight.max[2], 
                   edge.width.max = 12, 
                   title.name = paste0("Number of interactions - ", names(cellchat_list)[i]))
}
num.link <- sapply(cellchat_list, function(x) {rowSums(x@net$count) + colSums(x@net$count)-diag(x@net$count)})
weight.MinMax <- c(min(num.link), max(num.link)) 
gg <- list()
for (i in 1:length(cellchat_list)) {
  gg[[i]] <- netAnalysis_signalingRole_scatter(cellchat_list[[i]], title = names(cellchat_list)[i], weight.MinMax = weight.MinMax)
}
pdf("figures/cellchat/G1G2_rnrZ_senderincomg.pdf", width = 16 , height = 4.5 )
patchwork::wrap_plots(plots = gg)
dev.off()

pdf("figures/cellchat/pathways_G1G2.pdf",height = 20,width = 5)
rankNet(
  cellchat_merged,
  mode = "comparison",
  stacked = TRUE,
  do.stat = TRUE
)
dev.off()

png 
  2